<div align="center">

# 🟦 ParsiCoverTitle-VLM

### Fine-Tuning Qwen2.5-VL and Gemma 3 for Persian Book Cover Title Recognition

<br>

### **Prepared by: Ziba Dehghani**

<br>

## 🎯 Project Goal

Extract the main Persian book title from cover images using Vision-Language Models, Supervised Fine-Tuning, and LoRA adapters.

<br>

| 📚 Dataset | 🤖 Models | 📊 Evaluation |
|:---:|:---:|:---:|
| Persian Book Covers & Titles | Qwen2.5-VL-3B · Gemma-3-4B | Exact Match · Edit Similarity · CER |

<br>

## 🏆 Best Held-Out Test Result

**Gemma-3-4B Fine-Tuned** achieved the strongest overall performance:

| Exact Match | Edit Similarity | CER |
|:---:|:---:|:---:|
| **14.8%** | **55.16%** | **53.09%** |

<br>

<sub>Google Colab · NVIDIA L4 · Unsloth · TRL</sub>

</div>

## 1. Environment and Reproducibility

Experiments were conducted in **Google Colab** with an **NVIDIA L4 GPU** using Unsloth and TRL.

- Fine-tuning method: Supervised Fine-Tuning (SFT) with LoRA adapters
- Quantization: 4-bit
- Random seed: 42
- Inference: deterministic (`do_sample=False`)

The following cells install dependencies, mount Google Drive, and define the shared experiment configuration.


In [ ]:
# ----------------------------------------------------------------نصب کتابخانه‌ها---------------------------------------------------------

%%capture

!pip install -q --upgrade unsloth
!pip install -q "transformers>=4.56.2" "trl>=0.22.2"
!pip install -q datasets accelerate bitsandbytes peft
!pip install -q python-Levenshtein pandas matplotlib pillow tqdm

In [ ]:
# ----------------------------------------------------------import تنظیمات اصلی و----------------------------------------------------

import os
import re
import gc
import random
import shutil
import unicodedata
from io import BytesIO
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from PIL import Image as PILImage
from PIL import UnidentifiedImageError

from datasets import load_dataset, Dataset, Features, Image as HFImage, Value
from tqdm.auto import tqdm
from Levenshtein import distance as levenshtein_distance

In [ ]:
# ----------------------------------------------اتصال Google Drive  مسیرهای پروژه-------------------------------------------------

from google.colab import drive
drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/ParsiCoverTitle-VLM_Final")
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
CHECKPOINT_ROOT = PROJECT_ROOT / "checkpoints"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Output root:", OUTPUT_ROOT)
print("Checkpoint root:", CHECKPOINT_ROOT)

In [ ]:
# ------------------------------------------------------------------تنظیمات اصلی پروژه--------------------------------------------------

SEED = 42
DATASET_ID = "shenasa/bookroom-persian-book-covers-and-titles"

TRAIN_SIZE = 2000
VAL_SIZE = 250
TEST_SIZE = 250

MAX_IMAGE_SIDE = 512
MAX_NEW_TOKENS = 64
MAX_SEQ_LENGTH = 2048

NUM_TRAIN_EPOCHS = 2
TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 2e-4
WARMUP_STEPS = 25

DROP_TITLE_OVERLAP = True

EXPERIMENT_NAME = "final_l4_2000train_2epochs"

OUTPUT_ROOT = OUTPUT_ROOT / EXPERIMENT_NAME
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB",
    )

In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

## 2. Dataset Preparation and Data Splits

**Dataset:** `shenasa/bookroom-persian-book-covers-and-titles`

The task is to extract only the main Persian book title from a cover image. Samples with invalid images were excluded, and normalized title overlap was reduced between splits.

| Split | Samples | Purpose |
|---|---:|---|
| Train | 2,000 | Fine-tuning |
| Validation | 250 | Model selection and error analysis |
| Held-out Test | 250 | Final evaluation only |

The same underlying splits are used for Qwen and Gemma to ensure a fair comparison.


In [ ]:
# ------------------------------------------------------------------بارگذاری دیتاست-------------------------------------------------------

raw_ds = load_dataset(DATASET_ID)

print(raw_ds)
print("Train columns:", raw_ds["train"].column_names)
print("Test columns:", raw_ds["test"].column_names)

In [ ]:
# ---------------------------------------------------نرمال‌سازی عنوان های فارسی------------------------------------------------------------

PERSIAN_DIGITS = "۰۱۲۳۴۵۶۷۸۹"
ARABIC_DIGITS = "٠١٢٣٤٥٦٧٨٩"
LATIN_DIGITS = "0123456789"

DIGIT_TRANSLATION = str.maketrans(
    PERSIAN_DIGITS + ARABIC_DIGITS,
    LATIN_DIGITS + LATIN_DIGITS,
)

ARABIC_DIACRITICS = re.compile(r"[\u064B-\u065F\u0670\u06D6-\u06ED]")
PUNCTUATION = re.compile(r"""[!"#$%&'()*+,./:;<=>?@\[\\\]^_`{|}~،؛؟«»ـ…]""")

def normalize_title(text: str) -> str:
    if text is None:
        return ""

    text = unicodedata.normalize("NFKC", str(text))
    text = text.translate(DIGIT_TRANSLATION)

    text = (
        text.replace("ي", "ی")
            .replace("ى", "ی")
            .replace("ك", "ک")
            .replace("ة", "ه")
            .replace("ۀ", "ه")
            .replace("\u200c", " ")
    )

    text = ARABIC_DIACRITICS.sub("", text)
    text = PUNCTUATION.sub(" ", text)
    text = re.sub(r"\s+", " ", text).strip().lower()

    return text

In [ ]:
# ------------------------------------------------------خواندن صحیح تصویرهااگر خراب باشند None -----------------------------------------------

raw_train_bytes = raw_ds["train"].cast_column("image", HFImage(decode=False))
raw_test_bytes = raw_ds["test"].cast_column("image", HFImage(decode=False))

RESAMPLE_METHOD = getattr(PILImage, "Resampling", PILImage).LANCZOS

def decode_image_safely(image_record):
    try:
        if isinstance(image_record, dict):
            image_bytes = image_record.get("bytes")
            image_path = image_record.get("path")

            if image_bytes is not None:
                source = BytesIO(image_bytes)
            elif image_path:
                source = image_path
            else:
                return None
        else:
            source = image_record

        with PILImage.open(source) as image:
            image.load()
            image = image.convert("RGB")
            image.thumbnail((MAX_IMAGE_SIDE, MAX_IMAGE_SIDE), RESAMPLE_METHOD)
            return image.copy()

    except (UnidentifiedImageError, OSError, ValueError, TypeError):
        return None

In [ ]:
# ------------------------------------------------------------------splitهای train/validation/test----------------------------------------------------------------------

def collect_valid_records(
    dataset,
    target_size,
    seed,
    forbidden_titles=None,
    forbidden_source_ids=None,
):
    forbidden_titles = forbidden_titles or set()
    forbidden_source_ids = forbidden_source_ids or set()

    shuffled_dataset = dataset.shuffle(seed=seed)

    records = []
    skipped_bad_images = 0
    skipped_duplicate_titles = 0
    skipped_source_ids = 0
    scanned_rows = 0

    for row in shuffled_dataset:
        scanned_rows += 1

        source_id = row["_source_id"]
        normalized_title = normalize_title(row["text"])

        if not normalized_title:
            continue

        if source_id in forbidden_source_ids:
            skipped_source_ids += 1
            continue

        if normalized_title in forbidden_titles:
            skipped_duplicate_titles += 1
            continue

        image = decode_image_safely(row["image"])
        if image is None:
            skipped_bad_images += 1
            continue

        records.append(
            {
                "image": image,
                "text": str(row["text"]),
                "normalized_title": normalized_title,
                "source_id": source_id,
            }
        )

        if len(records) >= target_size:
            break

    if len(records) < target_size:
        raise RuntimeError(
            f"Only {len(records)} valid records were collected, "
            f"but {target_size} were needed."
        )

    audit = {
        "target_size": target_size,
        "collected": len(records),
        "scanned_rows": scanned_rows,
        "skipped_bad_images": skipped_bad_images,
        "skipped_duplicate_titles": skipped_duplicate_titles,
        "skipped_source_ids": skipped_source_ids,
    }

    return records, audit


valid_train_indices = [
    i for i, title in enumerate(raw_train_bytes["text"])
    if normalize_title(title)
]
valid_test_indices = [
    i for i, title in enumerate(raw_test_bytes["text"])
    if normalize_title(title)
]

train_metadata = raw_train_bytes.select(valid_train_indices)
test_metadata = raw_test_bytes.select(valid_test_indices)

train_metadata = train_metadata.add_column("_source_id", valid_train_indices)
test_metadata = test_metadata.add_column("_source_id", valid_test_indices)

train_records, train_audit = collect_valid_records(
    dataset=train_metadata,
    target_size=TRAIN_SIZE,
    seed=SEED,
)

train_title_set = {row["normalized_title"] for row in train_records}
train_source_id_set = {row["source_id"] for row in train_records}

val_records, val_audit = collect_valid_records(
    dataset=train_metadata,
    target_size=VAL_SIZE,
    seed=SEED + 1,
    forbidden_titles=train_title_set if DROP_TITLE_OVERLAP else set(),
    forbidden_source_ids=train_source_id_set,
)

validation_title_set = {row["normalized_title"] for row in val_records}
seen_title_set = train_title_set | validation_title_set

test_records, test_audit = collect_valid_records(
    dataset=test_metadata,
    target_size=TEST_SIZE,
    seed=SEED + 2,
    forbidden_titles=seen_title_set if DROP_TITLE_OVERLAP else set(),
)

In [ ]:
# ---------------------------------------------------------------------------قالب Dataset ---------------------------------------------

dataset_features = Features(
    {
        "image": HFImage(),
        "text": Value("string"),
    }
)

def records_to_dataset(records):
    return Dataset.from_dict(
        {
            "image": [row["image"] for row in records],
            "text": [row["text"] for row in records],
        },
        features=dataset_features,
    )

train_raw = records_to_dataset(train_records)
val_raw = records_to_dataset(val_records)
test_raw = records_to_dataset(test_records)

split_report = pd.DataFrame(
    [
        {"split": "train", "rows": len(train_raw), **train_audit},
        {"split": "validation", "rows": len(val_raw), **val_audit},
        {"split": "test", "rows": len(test_raw), **test_audit},
    ]
)

display(split_report)

In [ ]:
# --------------------------------------------- نمایش از جلدها و عنوان‌هایشان--------------------------------------------------------

def show_dataset_samples(dataset, n=6, seed=SEED):
    selected = dataset.shuffle(seed=seed).select(range(min(n, len(dataset))))

    fig, axes = plt.subplots(1, len(selected), figsize=(3 * len(selected), 5))
    if len(selected) == 1:
        axes = [axes]

    for ax, row in zip(axes, selected):
        ax.imshow(row["image"].convert("RGB"))
        ax.axis("off")

    plt.tight_layout()
    plt.show()

    display(pd.DataFrame({"عنوان صحیح": selected["text"]}))

show_dataset_samples(train_raw, n=6)

## 3. Prompt Design and Evaluation Metrics

The instruction explicitly requests **only the main title** and excludes author, translator, publisher, price, edition information, and promotional text.

| Metric | Meaning | Better |
|---|---|:---:|
| Exact Match | Prediction exactly equals the reference title | Higher |
| Edit Similarity | Character-level text similarity to the reference | Higher |
| Character Error Rate (CER) | Character-level transcription error | Lower |


In [ ]:
# --------------------------------------------------------------Prompt--------------------------------------------------------

SYSTEM_MESSAGE = (
    "تو یک سامانه دقیق OCR برای جلد کتاب‌های فارسی هستی. "
    "وظیفه‌ات فقط خواندن عنوان اصلی کتاب از روی تصویر جلد است."
)

TITLE_INSTRUCTION = """
عنوان اصلی کتاب را از روی تصویر جلد استخراج کن.

قواعد پاسخ:
- فقط عنوان اصلی کتاب را بنویس.
- نام نویسنده، مترجم، ناشر، قیمت، شماره چاپ، شعار تبلیغاتی یا توضیح اضافه را ننویس.
- هیچ مقدمه، جمله، برچسبی مانند «عنوان کتاب:» یا علامت نقل‌قول اضافه نکن.
- اگر عنوان چند بخش دارد، آن را کامل و با همان ترتیب بنویس.
""".strip()

print(TITLE_INSTRUCTION)

In [ ]:
# --------------------------------------------------------تعریف معیارها----------------------------------------

def postprocess_prediction(text: str) -> str:
    """
    پیشوندها، برچسب‌های اضافی و خط‌های اضافه را از خروجی مدل حذف می‌کند.
    """

    text = str(text).strip()

    text = re.sub(
        r"^\s*(عنوان\s*(اصلی\s*)?(کتاب\s*)?[:：\-–—]?\s*)",
        "",
        text,
        flags=re.IGNORECASE,
    )

    text = re.sub(
        r"^\s*(book\s*title\s*[:：\-–—]?\s*)",
        "",
        text,
        flags=re.IGNORECASE,
    )

    text = text.strip().strip("«»\"'")

    return text.split("\n")[0].strip()


def normalized_edit_similarity(reference: str, prediction: str) -> float:
    """
    شباهت کاراکتری نرمال‌شده بین 0 و 1.
    مقدار 1 یعنی تطابق کامل.
    """

    reference = normalize_title(reference)
    prediction = normalize_title(prediction)

    maximum_length = max(len(reference), len(prediction), 1)

    return 1 - (
        levenshtein_distance(reference, prediction) / maximum_length
    )


def character_error_rate(reference: str, prediction: str) -> float:
    """
    نرخ خطای کاراکتری.
    مقدار کمتر بهتر است و صفر یعنی تطابق کامل.
    """

    reference = normalize_title(reference)
    prediction = normalize_title(prediction)

    return levenshtein_distance(
        reference,
        prediction,
    ) / max(len(reference), 1)


def calculate_metrics(results_df: pd.DataFrame) -> dict:
    """
    معیارهای نهایی را از DataFrame پیش‌بینی‌ها محاسبه می‌کند.
    """

    return {
        "exact_match": results_df["exact_match"].mean(),
        "exact_match_percent": (
            results_df["exact_match"].mean() * 100
        ),
        "mean_edit_similarity": (
            results_df["edit_similarity"].mean()
        ),
        "mean_edit_similarity_percent": (
            results_df["edit_similarity"].mean() * 100
        ),
        "mean_character_error_rate": (
            results_df["character_error_rate"].mean()
        ),
        "mean_character_error_rate_percent": (
            results_df["character_error_rate"].mean() * 100
        ),
    }


print("Evaluation utilities are ready.")

In [ ]:
# --------------------------------------------------------------------حافظه GPU---------------------------------------------------------------------

def show_gpu_memory():
    total_gb = (
        torch.cuda.get_device_properties(0).total_memory / 1024**3
    )
    allocated_gb = torch.cuda.memory_allocated() / 1024**3
    reserved_gb = torch.cuda.memory_reserved() / 1024**3

    print(f"GPU total memory: {total_gb:.2f} GB")
    print(f"Allocated memory: {allocated_gb:.2f} GB")
    print(f"Reserved memory: {reserved_gb:.2f} GB")


show_gpu_memory()

In [ ]:
# -------------------------------------------------تبدیل به قالب Qwen-------------------------------------------------

def convert_to_qwen_conversation(sample):
    """
    هر نمونه را به قالب گفت‌وگویی موردنیاز Qwen2.5-VL تبدیل می‌کند.
    """

    return {
        "messages": [
            {
                "role": "system",
                "content": [
                    {
                        "type": "text",
                        "text": SYSTEM_MESSAGE,
                    }
                ],
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "image": sample["image"],
                    },
                    {
                        "type": "text",
                        "text": TITLE_INSTRUCTION,
                    },
                ],
            },
            {
                "role": "assistant",
                "content": [
                    {
                        "type": "text",
                        "text": sample["text"],
                    }
                ],
            },
        ]
    }


qwen_train_dataset = [
    convert_to_qwen_conversation(sample)
    for sample in train_raw
]

qwen_val_dataset = [
    convert_to_qwen_conversation(sample)
    for sample in val_raw
]

print("Qwen train conversations:", len(qwen_train_dataset))
print("Qwen validation conversations:", len(qwen_val_dataset))

In [ ]:
# ------------------------------------------------------------بررسی یک نمونه-------------------------------------------------------

qwen_example = qwen_train_dataset[0]["messages"]

print("System message:")
print(qwen_example[0]["content"][0]["text"])

print("\nInstruction:")
print(qwen_example[1]["content"][1]["text"])

print("\nCorrect title:")
print(qwen_example[2]["content"][0]["text"])

display(qwen_example[1]["content"][0]["image"])

## 4. Qwen2.5-VL-3B: Baseline, Fine-Tuning, and Validation Analysis

This section evaluates the base Qwen model on the validation split, applies LoRA-based SFT, and compares the resulting fine-tuned model with the baseline.


In [ ]:
# ---------------------------------------------------------بارگذاری Qwen2.5-VL-3B-------------------------------------------------

from unsloth import FastVisionModel

QWEN_MODEL_ID = "unsloth/Qwen2.5-VL-3B-Instruct-bnb-4bit"

qwen_model, qwen_processor = FastVisionModel.from_pretrained(
    model_name=QWEN_MODEL_ID,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
    max_seq_length=MAX_SEQ_LENGTH,
)

print("Qwen2.5-VL-3B loaded successfully.")

show_gpu_memory()

In [ ]:
# -------------------------------------------------------تابع استخراج عنوان با Qwen پایه---------------------------------------------------------------

def generate_book_title_qwen(
    model,
    processor,
    image,
    max_new_tokens=MAX_NEW_TOKENS,
):
    """
    عنوان کتاب را از روی تصویر جلد با Qwen تولید می‌کند.
    """

    messages = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": SYSTEM_MESSAGE,
                }
            ],
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                },
                {
                    "type": "text",
                    "text": TITLE_INSTRUCTION,
                },
            ],
        },
    ]

    input_text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    model_inputs = processor(
        image.convert("RGB"),
        input_text,
        add_special_tokens=False,
        return_tensors="pt",
    ).to("cuda")

    with torch.inference_mode():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
        )

    prompt_length = model_inputs["input_ids"].shape[1]
    generated_ids = generated_ids[:, prompt_length:]

    generated_text = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    return generated_text.strip()

In [ ]:
# ---------------------------------------------------تست قبل از Fine-Tuning-------------------------------------------------------

FastVisionModel.for_inference(qwen_model)

demo_index = 0

qwen_demo_image = val_raw[demo_index]["image"]
qwen_demo_title = val_raw[demo_index]["text"]

qwen_baseline_demo_prediction = generate_book_title_qwen(
    model=qwen_model,
    processor=qwen_processor,
    image=qwen_demo_image,
)

plt.figure(figsize=(5, 7))
plt.imshow(qwen_demo_image.convert("RGB"))
plt.axis("off")
plt.show()

print("عنوان واقعی:")
print(qwen_demo_title)

print("\nپیش‌بینی Qwen پایه:")
print(qwen_baseline_demo_prediction)

In [ ]:
# -----------------------------------------------------------تابع ارزیابی---------------------------------------------------------

def evaluate_title_model(
    model,
    processor,
    dataset,
    split_name,
    model_label,
    generate_function,
):
    """
    مدل را روی یک split ارزیابی می‌کند و
    پیش‌بینی‌ها و معیارهای نهایی را برمی‌گرداند.
    """

    rows = []

    for index, sample in tqdm(
        enumerate(dataset),
        total=len(dataset),
        desc=f"Evaluating {model_label} on {split_name}",
    ):
        raw_prediction = generate_function(
            model=model,
            processor=processor,
            image=sample["image"],
        )

        clean_prediction = postprocess_prediction(raw_prediction)
        reference = sample["text"]

        reference_normalized = normalize_title(reference)
        prediction_normalized = normalize_title(clean_prediction)

        rows.append(
            {
                "index": index,
                "split": split_name,
                "model": model_label,
                "reference_raw": reference,
                "prediction_raw": raw_prediction,
                "prediction_clean": clean_prediction,
                "reference_normalized": reference_normalized,
                "prediction_normalized": prediction_normalized,
                "edit_similarity": normalized_edit_similarity(
                    reference,
                    clean_prediction,
                ),
                "character_error_rate": character_error_rate(
                    reference,
                    clean_prediction,
                ),
                "exact_match": (
                    reference_normalized == prediction_normalized
                ),
            }
        )

    results_df = pd.DataFrame(rows)
    metrics = calculate_metrics(results_df)

    return results_df, metrics

In [ ]:
# ---------------------------------------------------ارزیابی Qwen پایه روی validation---------------------------------------------

FastVisionModel.for_inference(qwen_model)

qwen_baseline_val_df, qwen_baseline_val_metrics = (
    evaluate_title_model(
        model=qwen_model,
        processor=qwen_processor,
        dataset=val_raw,
        split_name="validation",
        model_label="Qwen2.5-VL-3B-base",
        generate_function=generate_book_title_qwen,
    )
)

display(pd.DataFrame([qwen_baseline_val_metrics]))

qwen_baseline_val_path = (
    OUTPUT_ROOT / "qwen_baseline_validation_predictions.csv"
)

qwen_baseline_val_df.to_csv(
    qwen_baseline_val_path,
    index=False,
    encoding="utf-8-sig",
)

print(f"Saved baseline predictions to: {qwen_baseline_val_path}")

In [ ]:
# ----------------------------------------------------نمایش خطاهای Qwen پایه------------------------------------------------------

def show_error_gallery(
    results_df,
    dataset,
    n=6,
    figure_title="Qwen Base: Lowest-Similarity Validation Errors",
):
    """
    بدترین پیش‌بینی‌ها را بر اساس کمترین شباهت کاراکتری
    همراه با تصویر جلد نمایش می‌دهد.
    """

    selected_df = (
        results_df
        .sort_values(
            by=["edit_similarity", "character_error_rate"],
            ascending=[True, False],
        )
        .head(n)
        .copy()
    )

    columns = 3
    rows = int(np.ceil(len(selected_df) / columns))

    fig, axes = plt.subplots(
        rows,
        columns,
        figsize=(18, 6 * rows),
    )

    axes = np.array(axes).reshape(-1)

    for axis, (_, row) in zip(axes, selected_df.iterrows()):
        sample_index = int(row["index"])
        image = dataset[sample_index]["image"].convert("RGB")

        axis.imshow(image)
        axis.axis("off")

        axis.set_title(
            (
                f"Validation index: {sample_index}\n"
                f"Similarity: {row['edit_similarity']:.2f}\n"
                f"CER: {row['character_error_rate']:.2f}"
            ),
            fontsize=11,
        )

    for axis in axes[len(selected_df):]:
        axis.axis("off")

    plt.suptitle(
        figure_title,
        fontsize=18,
        y=1.02,
    )

    plt.tight_layout()
    plt.show()

    display(
        selected_df[
            [
                "index",
                "reference_raw",
                "prediction_raw",
                "prediction_clean",
                "edit_similarity",
                "character_error_rate",
            ]
        ].reset_index(drop=True)
    )

    return selected_df


qwen_baseline_errors_df = (
    qwen_baseline_val_df[
        ~qwen_baseline_val_df["exact_match"]
    ]
    .sort_values(
        by=["edit_similarity", "character_error_rate"],
        ascending=[True, False],
    )
    .copy()
)

qwen_baseline_errors_path = (
    OUTPUT_ROOT / "qwen_baseline_validation_errors.csv"
)

qwen_baseline_errors_df.to_csv(
    qwen_baseline_errors_path,
    index=False,
    encoding="utf-8-sig",
)

qwen_baseline_error_gallery_df = show_error_gallery(
    results_df=qwen_baseline_val_df,
    dataset=val_raw,
    n=6,
    figure_title="Qwen2.5-VL-3B Base: Validation Errors Before Fine-Tuning",
)

print(f"Saved baseline errors to: {qwen_baseline_errors_path}")

In [ ]:
# ---------------------------------------------------------------افزودن LoRA--------------------------------------------------

from peft import PeftModel

if isinstance(qwen_model, PeftModel):
    print("Qwen already has LoRA adapters. Reusing the existing adapter.")
else:
    FastVisionModel.for_training(qwen_model)

    qwen_model = FastVisionModel.get_peft_model(
        qwen_model,

        finetune_vision_layers=True,
        finetune_language_layers=True,
        finetune_attention_modules=True,
        finetune_mlp_modules=True,

        r=8,
        lora_alpha=8,
        lora_dropout=0.05,
        bias="none",

        random_state=SEED,
        target_modules="all-linear",

        use_rslora=False,
        loftq_config=None,
    )

qwen_model.print_trainable_parameters()

In [ ]:
# -----------------------------------------------------بررسی قالب مکالمه --------------------------------------------------------------

qwen_template_preview = qwen_processor.apply_chat_template(
    qwen_train_dataset[0]["messages"],
    tokenize=False,
)

print(repr(qwen_template_preview[-1000:]))

In [ ]:
# -------------------------------------------Data Collator برای آموزش پاسخ‌محور--------------------------------------------------------

from unsloth.trainer import UnslothVisionDataCollator

QWEN_INSTRUCTION_PART = "<|im_start|>user\n"
QWEN_RESPONSE_PART = "<|im_start|>assistant\n"

qwen_data_collator = UnslothVisionDataCollator(
    qwen_model,
    qwen_processor,

    train_on_responses_only=True,
    instruction_part=QWEN_INSTRUCTION_PART,
    response_part=QWEN_RESPONSE_PART,

    force_match=True,
    completion_only_loss=True,
    resize="max",
)

print("Qwen vision data collator is ready.")

In [ ]:
# -------------------------------------------------کنترل Labelها -----------------------------------------------------------------------

debug_qwen_batch = qwen_data_collator(
    [
        qwen_train_dataset[0],
        qwen_train_dataset[1],
    ]
)

print("Batch keys:", list(debug_qwen_batch.keys()))
print("input_ids shape:", debug_qwen_batch["input_ids"].shape)
print("labels shape:", debug_qwen_batch["labels"].shape)

qwen_active_label_tokens = (
    debug_qwen_batch["labels"][0] != -100
).sum().item()

print(
    "Number of tokens contributing to loss:",
    qwen_active_label_tokens,
)

assert qwen_active_label_tokens > 0, (
    "No assistant-response tokens were selected for loss. "
    "Do not start training until this is fixed."
)

In [ ]:
# ----------------------------------------------ساخت Trainer نهایی Qwen---------------------------------------------------------------

from trl import SFTConfig, SFTTrainer

QWEN_OUTPUT_DIR = CHECKPOINT_ROOT / "qwen25vl3b_training"
QWEN_FINAL_ADAPTER_DIR = (
    OUTPUT_ROOT / "qwen25vl3b_persian_book_title_lora"
)

USE_BF16 = torch.cuda.is_bf16_supported()

qwen_training_args = SFTConfig(
    output_dir=str(QWEN_OUTPUT_DIR),

    # آموزش
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    # بهینه‌سازی
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    weight_decay=0.01,
    max_grad_norm=0.3,

    # کاهش مصرف حافظه
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False,
    },

    # دقت محاسبات، متناسب با L4
    fp16=not USE_BF16,
    bf16=USE_BF16,

    # ذخیره‌سازی مقاوم در برابر قطع Colab
    logging_steps=10,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=2,
    report_to="none",

    # ضروری برای داده‌ی تصویر + متن
    remove_unused_columns=False,
    dataset_text_field="",
    dataset_kwargs={
        "skip_prepare_dataset": True,
    },
    max_length=MAX_SEQ_LENGTH,

    seed=SEED,
)

qwen_trainer = SFTTrainer(
    model=qwen_model,
    args=qwen_training_args,
    train_dataset=qwen_train_dataset,
    data_collator=qwen_data_collator,
    processing_class=qwen_processor.tokenizer,
)

print("Qwen trainer is ready.")
print("Training examples:", len(qwen_train_dataset))
print(
    "Effective batch size:",
    TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
)
print("Total planned optimizer steps: about 500")
print("Using BF16:", USE_BF16)
print("Checkpoint directory:", QWEN_OUTPUT_DIR)

In [ ]:
# ----------------------------------------------------------آموزش Qwen---------------------------------------------------------

def get_latest_checkpoint(checkpoint_dir: Path):
    checkpoints = list(checkpoint_dir.glob("checkpoint-*"))

    if not checkpoints:
        return None

    return max(
        checkpoints,
        key=lambda path: int(path.name.split("-")[-1]),
    )


latest_qwen_checkpoint = get_latest_checkpoint(QWEN_OUTPUT_DIR)

FastVisionModel.for_training(qwen_model)

if latest_qwen_checkpoint is None:
    print("No checkpoint found. Starting Qwen training from step 0.")

    qwen_train_result = qwen_trainer.train()
else:
    print(f"Resuming Qwen training from: {latest_qwen_checkpoint}")

    qwen_train_result = qwen_trainer.train(
        resume_from_checkpoint=str(latest_qwen_checkpoint),
    )

print(qwen_train_result)

In [ ]:
qwen_model.save_pretrained(
    str(QWEN_FINAL_ADAPTER_DIR)
)

qwen_processor.save_pretrained(
    str(QWEN_FINAL_ADAPTER_DIR)
)

qwen_history_df = pd.DataFrame(
    qwen_trainer.state.log_history
)

qwen_history_path = (
    OUTPUT_ROOT / "qwen_training_history.csv"
)

qwen_history_df.to_csv(
    qwen_history_path,
    index=False,
    encoding="utf-8-sig",
)

print(f"Saved final Qwen LoRA adapter to: {QWEN_FINAL_ADAPTER_DIR}")
print(f"Saved training history to: {qwen_history_path}")

In [ ]:
# ---------------------------------------------------نمودار Loss آموزش Qwen---------------------------------------

qwen_loss_history_df = qwen_history_df.dropna(
    subset=["loss"]
).copy()

plt.figure(figsize=(10, 5))

plt.plot(
    qwen_loss_history_df["step"],
    qwen_loss_history_df["loss"],
    marker="o",
)

plt.xlabel("Training step")
plt.ylabel("Training loss")
plt.title("Qwen2.5-VL-3B Training Loss")

plt.grid(True)
plt.tight_layout()

qwen_loss_plot_path = (
    OUTPUT_ROOT / "qwen_training_loss.png"
)

plt.savefig(
    qwen_loss_plot_path,
    dpi=200,
    bbox_inches="tight",
)

plt.show()

print(f"Saved training-loss chart to: {qwen_loss_plot_path}")

In [ ]:
# -------------------------------------------------ارزیابی Qwen Fine-Tuned روی validation-------------------------------------------

FastVisionModel.for_inference(qwen_model)

qwen_finetuned_val_df, qwen_finetuned_val_metrics = (
    evaluate_title_model(
        model=qwen_model,
        processor=qwen_processor,
        dataset=val_raw,
        split_name="validation",
        model_label="Qwen2.5-VL-3B-finetuned",
        generate_function=generate_book_title_qwen,
    )
)

display(pd.DataFrame([qwen_finetuned_val_metrics]))

qwen_finetuned_val_path = (
    OUTPUT_ROOT / "qwen_finetuned_validation_predictions.csv"
)

qwen_finetuned_val_df.to_csv(
    qwen_finetuned_val_path,
    index=False,
    encoding="utf-8-sig",
)

print(
    "Saved fine-tuned validation predictions to: "
    f"{qwen_finetuned_val_path}"
)

In [ ]:
qwen_validation_comparison_df = pd.DataFrame(
    [
        {
            "Model": "Qwen2.5-VL-3B Base",
            **qwen_baseline_val_metrics,
        },
        {
            "Model": "Qwen2.5-VL-3B Fine-Tuned",
            **qwen_finetuned_val_metrics,
        },
    ]
)

base_row = qwen_validation_comparison_df.iloc[0]

qwen_validation_comparison_df["Exact Match Change (pp)"] = (
    qwen_validation_comparison_df["exact_match_percent"]
    - base_row["exact_match_percent"]
)

qwen_validation_comparison_df["Edit Similarity Change (pp)"] = (
    qwen_validation_comparison_df["mean_edit_similarity_percent"]
    - base_row["mean_edit_similarity_percent"]
)

qwen_validation_comparison_df["CER Change (pp)"] = (
    qwen_validation_comparison_df[
        "mean_character_error_rate_percent"
    ]
    - base_row["mean_character_error_rate_percent"]
)

display(qwen_validation_comparison_df)

qwen_validation_comparison_path = (
    OUTPUT_ROOT / "qwen_validation_comparison.csv"
)

qwen_validation_comparison_df.to_csv(
    qwen_validation_comparison_path,
    index=False,
    encoding="utf-8-sig",
)

print(
    "Saved Qwen validation comparison to: "
    f"{qwen_validation_comparison_path}"
)

In [ ]:
# ------------------------------------------------------گالری خطاهای Qwen بعد از Fine-Tuning---------------------------------------------------

qwen_finetuned_errors_df = (
    qwen_finetuned_val_df[
        ~qwen_finetuned_val_df["exact_match"]
    ]
    .sort_values(
        by=["edit_similarity", "character_error_rate"],
        ascending=[True, False],
    )
    .copy()
)

qwen_finetuned_errors_path = (
    OUTPUT_ROOT / "qwen_finetuned_validation_errors.csv"
)

qwen_finetuned_errors_df.to_csv(
    qwen_finetuned_errors_path,
    index=False,
    encoding="utf-8-sig",
)

qwen_finetuned_error_gallery_df = show_error_gallery(
    results_df=qwen_finetuned_val_df,
    dataset=val_raw,
    n=6,
    figure_title=(
        "Qwen2.5-VL-3B Fine-Tuned: "
        "Validation Errors After Fine-Tuning"
    ),
)

print(
    "Saved fine-tuned errors to: "
    f"{qwen_finetuned_errors_path}"
)

In [ ]:
# ------------------------------------------نمونه‌های بهبود‌یافته قبل و بعد-----------------------------------------

qwen_before_after_df = qwen_baseline_val_df.merge(
    qwen_finetuned_val_df,
    on="index",
    suffixes=("_base", "_finetuned"),
)

qwen_before_after_df["similarity_gain"] = (
    qwen_before_after_df["edit_similarity_finetuned"]
    - qwen_before_after_df["edit_similarity_base"]
)

qwen_before_after_df["cer_reduction"] = (
    qwen_before_after_df["character_error_rate_base"]
    - qwen_before_after_df["character_error_rate_finetuned"]
)

qwen_before_after_df = qwen_before_after_df.sort_values(
    by=["similarity_gain", "cer_reduction"],
    ascending=[False, False],
).copy()

display(
    qwen_before_after_df[
        [
            "index",
            "reference_raw_base",
            "prediction_clean_base",
            "prediction_clean_finetuned",
            "edit_similarity_base",
            "edit_similarity_finetuned",
            "similarity_gain",
            "character_error_rate_base",
            "character_error_rate_finetuned",
        ]
    ]
    .head(10)
    .reset_index(drop=True)
)

In [ ]:
# ------------------------------------------------------گالری تصویری بهترین بهبودهای Qwen------------------------------------------------------

def show_before_after_gallery(
    comparison_df,
    dataset,
    n=6,
    figure_title=(
        "Qwen2.5-VL-3B: Best Validation Improvements After Fine-Tuning"
    ),
):
    """
    نمونه‌هایی را نمایش می‌دهد که Fine-Tuning در آن‌ها
    بیشترین بهبود شباهت کاراکتری ایجاد کرده است.
    """

    selected_df = (
        comparison_df
        .sort_values(
            by=["similarity_gain", "cer_reduction"],
            ascending=[False, False],
        )
        .head(n)
        .copy()
    )

    columns = 3
    rows = int(np.ceil(len(selected_df) / columns))

    fig, axes = plt.subplots(
        rows,
        columns,
        figsize=(18, 6 * rows),
    )

    axes = np.array(axes).reshape(-1)

    for axis, (_, row) in zip(axes, selected_df.iterrows()):
        sample_index = int(row["index"])
        image = dataset[sample_index]["image"].convert("RGB")

        axis.imshow(image)
        axis.axis("off")

        axis.set_title(
            (
                f"Validation index: {sample_index}\n"
                f"Base similarity: {row['edit_similarity_base']:.2f}\n"
                f"Fine-Tuned similarity: "
                f"{row['edit_similarity_finetuned']:.2f}\n"
                f"Gain: {row['similarity_gain']:+.2f}"
            ),
            fontsize=11,
        )

    for axis in axes[len(selected_df):]:
        axis.axis("off")

    plt.suptitle(
        figure_title,
        fontsize=18,
        y=1.02,
    )

    plt.tight_layout()
    plt.show()

    result_table = selected_df[
        [
            "index",
            "reference_raw_base",
            "prediction_clean_base",
            "prediction_clean_finetuned",
            "edit_similarity_base",
            "edit_similarity_finetuned",
            "similarity_gain",
            "character_error_rate_base",
            "character_error_rate_finetuned",
        ]
    ].copy()

    result_table.columns = [
        "index",
        "عنوان واقعی",
        "پیش‌بینی مدل پایه",
        "پیش‌بینی Fine-Tuned",
        "شباهت پایه",
        "شباهت Fine-Tuned",
        "میزان بهبود شباهت",
        "CER پایه",
        "CER Fine-Tuned",
    ]

    display(result_table.reset_index(drop=True))

    return selected_df

In [ ]:
qwen_best_improvements_df = show_before_after_gallery(
    comparison_df=qwen_before_after_df,
    dataset=val_raw,
    n=6,
)

qwen_best_improvements_path = (
    OUTPUT_ROOT / "qwen_best_validation_improvements.csv"
)

qwen_best_improvements_df.to_csv(
    qwen_best_improvements_path,
    index=False,
    encoding="utf-8-sig",
)

print(
    "Saved Qwen best validation improvements to: "
    f"{qwen_best_improvements_path}"
)

In [ ]:
# -----------------------------------------------نمودار مقایسه Qwen روی Validation---------------------------------------------

qwen_metric_plot_df = pd.DataFrame(
    {
        "Metric": [
            "Exact Match",
            "Edit Similarity",
            "CER",
        ],
        "Base": [
            qwen_baseline_val_metrics["exact_match_percent"],
            qwen_baseline_val_metrics["mean_edit_similarity_percent"],
            qwen_baseline_val_metrics["mean_character_error_rate_percent"],
        ],
        "Fine-Tuned": [
            qwen_finetuned_val_metrics["exact_match_percent"],
            qwen_finetuned_val_metrics["mean_edit_similarity_percent"],
            qwen_finetuned_val_metrics["mean_character_error_rate_percent"],
        ],
    }
)

x = np.arange(len(qwen_metric_plot_df))
width = 0.28

plt.figure(figsize=(8, 4.6))

base_bars = plt.bar(
    x - width / 2,
    qwen_metric_plot_df["Base"],
    width,
    label="Base",
    color="#93C5FD",   # آبی ملایم
)

finetuned_bars = plt.bar(
    x + width / 2,
    qwen_metric_plot_df["Fine-Tuned"],
    width,
    label="Fine-Tuned",
    color="#F9A8D4",   # صورتی ملایم
)

plt.xticks(x, qwen_metric_plot_df["Metric"], fontsize=10)
plt.ylabel("Score (%)", fontsize=10)
plt.title(
    "Qwen2.5-VL-3B Validation: Base vs Fine-Tuned",
    fontsize=12,
)
plt.legend(frameon=False, fontsize=9)

plt.ylim(0, max(qwen_metric_plot_df["Fine-Tuned"].max(), qwen_metric_plot_df["Base"].max()) + 8)

for bars in [base_bars, finetuned_bars]:
    for bar in bars:
        height = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            height + 0.8,
            f"{height:.1f}",
            ha="center",
            va="bottom",
            fontsize=9,
        )

plt.tight_layout()

qwen_validation_plot_path = (
    OUTPUT_ROOT / "qwen_validation_metrics_compact_soft.png"
)

plt.savefig(
    qwen_validation_plot_path,
    dpi=200,
    bbox_inches="tight",
)

plt.show()

print(f"Saved compact chart to: {qwen_validation_plot_path}")

In [ ]:
def clear_gpu_objects(variable_names):
    """
    فقط اشیای سنگین مرتبط با مدل را از حافظه حذف می‌کند.
    نتایج، CSVها و DataFrameهای Qwen باقی می‌مانند.
    """

    for variable_name in variable_names:
        if variable_name in globals():
            del globals()[variable_name]

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

    print("Qwen model objects were cleared from GPU memory.")


clear_gpu_objects(
    [
        "qwen_model",
        "qwen_processor",
        "qwen_trainer",
        "qwen_data_collator",
        "debug_qwen_batch",
    ]
)

show_gpu_memory()

## 5. Gemma-3-4B: Baseline, Fine-Tuning, and Validation Analysis

This section follows the same protocol for Gemma: baseline validation, LoRA-based SFT, and validation-level error analysis.

> **Checkpoint recovery:** the training cell automatically resumes from the latest saved checkpoint when one is available.


In [ ]:
# -------------------------------------------------بارگذاری Gemma 3 4B-------------------------------------------

from unsloth import FastVisionModel, get_chat_template

GEMMA_MODEL_ID = "unsloth/gemma-3-4b-it-unsloth-bnb-4bit"

GEMMA_OUTPUT_DIR = (
    CHECKPOINT_ROOT
    / EXPERIMENT_NAME
    / "gemma3_4b_training"
)

GEMMA_FINAL_ADAPTER_DIR = (
    OUTPUT_ROOT
    / "gemma3_4b_persian_book_title_lora"
)

gemma_model, gemma_processor = FastVisionModel.from_pretrained(
    model_name=GEMMA_MODEL_ID,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
    max_seq_length=MAX_SEQ_LENGTH,
)

gemma_processor = get_chat_template(
    gemma_processor,
    "gemma-3",
)

show_gpu_memory()

print(f"Gemma model: {GEMMA_MODEL_ID}")
print(f"Checkpoint directory: {GEMMA_OUTPUT_DIR}")
print(f"Final adapter directory: {GEMMA_FINAL_ADAPTER_DIR}")

In [ ]:
# --------------------------------------ساخت دیتاست مکالمه‌ای Gemma---------------------------------

def make_gemma_from_qwen_example(example):
    """
    داده‌ی از قبل آماده‌شده برای Qwen را به قالب مکالمه‌ای Gemma تبدیل می‌کند.
    پیام system حذف می‌شود؛ تصویر، دستور و عنوان صحیح بدون تغییر باقی می‌مانند.
    """

    user_message = next(
        message
        for message in example["messages"]
        if message["role"] == "user"
    )

    assistant_message = next(
        message
        for message in example["messages"]
        if message["role"] == "assistant"
    )

    return {
        "messages": [
            user_message,
            assistant_message,
        ]
    }


gemma_train_dataset = Dataset.from_list(
    [
        make_gemma_from_qwen_example(example)
        for example in qwen_train_dataset
    ]
)

gemma_val_dataset = Dataset.from_list(
    [
        make_gemma_from_qwen_example(example)
        for example in qwen_val_dataset
    ]
)

print(f"Gemma train rows: {len(gemma_train_dataset)}")
print(f"Gemma validation rows: {len(gemma_val_dataset)}")

display(gemma_train_dataset[0])

In [ ]:
# --------------------------------------------تابع پیش‌بینی Gemma و تست یک نمونه---------------------------------------------------

@torch.inference_mode()
def generate_book_title_gemma(
    model,
    processor,
    image,
    max_new_tokens=MAX_NEW_TOKENS,
):
    """
    عنوان کتاب را از روی تصویر جلد با Gemma تولید می‌کند.
    """

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                },
                {
                    "type": "text",
                    "text": TITLE_INSTRUCTION,
                },
            ],
        }
    ]

    prompt = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
    )

    inputs = processor(
        images=[image.convert("RGB")],
        text=[prompt],
        return_tensors="pt",
        padding=True,
    ).to("cuda")

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        use_cache=True,
    )

    new_token_ids = generated_ids[
        :,
        inputs["input_ids"].shape[1]:,
    ]

    raw_prediction = processor.batch_decode(
        new_token_ids,
        skip_special_tokens=True,
    )[0]

    return postprocess_prediction(raw_prediction)


sample_index = 0
sample_example = qwen_val_dataset[sample_index]

sample_image = next(
    item["image"]
    for message in sample_example["messages"]
    if message["role"] == "user"
    for item in message["content"]
    if item["type"] == "image"
).convert("RGB")

reference_title = next(
    item["text"]
    for message in sample_example["messages"]
    if message["role"] == "assistant"
    for item in message["content"]
    if item["type"] == "text"
)

print("Reference title:")
print(reference_title)

print("\nGemma base prediction:")
print(
    generate_book_title_gemma(
        model=gemma_model,
        processor=gemma_processor,
        image=sample_image,
    )
)

In [ ]:
# ------------------------------------------ارزیابی Gemma پایه روی Validation-------------------------------------------

FastVisionModel.for_inference(gemma_model)

gemma_baseline_val_df, gemma_baseline_val_metrics = (
    evaluate_title_model(
        model=gemma_model,
        processor=gemma_processor,
        dataset=val_raw,
        split_name="validation",
        model_label="Gemma-3-4B-Base",
        generate_function=generate_book_title_gemma,
    )
)

display(pd.DataFrame([gemma_baseline_val_metrics]))

gemma_baseline_val_path = (
    OUTPUT_ROOT / "gemma_baseline_validation_predictions.csv"
)

gemma_baseline_val_df.to_csv(
    gemma_baseline_val_path,
    index=False,
    encoding="utf-8-sig",
)

print(
    "Saved Gemma baseline validation predictions to: "
    f"{gemma_baseline_val_path}"
)

In [ ]:
# ------------------------------------------گالری خطاهای Gemma پایه روی Validation------------------------------------------------------

gemma_baseline_errors_df = (
    gemma_baseline_val_df[
        ~gemma_baseline_val_df["exact_match"]
    ]
    .sort_values(
        by=["edit_similarity", "character_error_rate"],
        ascending=[True, False],
    )
    .copy()
)

gemma_baseline_errors_path = (
    OUTPUT_ROOT / "gemma_baseline_validation_errors.csv"
)

gemma_baseline_errors_df.to_csv(
    gemma_baseline_errors_path,
    index=False,
    encoding="utf-8-sig",
)

gemma_baseline_error_gallery_df = show_error_gallery(
    results_df=gemma_baseline_val_df,
    dataset=val_raw,
    n=6,
    figure_title=(
        "Gemma-3-4B Base: Validation Errors Before Fine-Tuning"
    ),
)

print(
    "Saved Gemma baseline errors to: "
    f"{gemma_baseline_errors_path}"
)

In [ ]:
# ---------------------------------------------افزودن LoRA به Gemma برای Fine-Tuning-----------------------------------------------

from peft import PeftModel

if isinstance(gemma_model, PeftModel):
    print("LoRA adapter is already attached to Gemma.")
else:
    FastVisionModel.for_training(gemma_model)

    gemma_model = FastVisionModel.get_peft_model(
        gemma_model,
        finetune_vision_layers=True,
        finetune_language_layers=True,
        finetune_attention_modules=True,
        finetune_mlp_modules=True,
        r=8,
        lora_alpha=8,
        lora_dropout=0.05,
        bias="none",
        random_state=SEED,
        target_modules="all-linear",
        use_rslora=False,
        loftq_config=None,
    )

gemma_model.print_trainable_parameters()

In [ ]:
gemma_rendered_example = gemma_processor.apply_chat_template(
    gemma_train_dataset[0]["messages"],
    tokenize=False,
)

print(gemma_rendered_example)

In [ ]:
# -------------------------------------------ساخت Data Collator و بررسی Loss Masking برای Gemma---------------------------------------

from unsloth.trainer import UnslothVisionDataCollator

GEMMA_INSTRUCTION_PART = "<start_of_turn>user\n"
GEMMA_RESPONSE_PART = "<start_of_turn>model\n"

gemma_data_collator = UnslothVisionDataCollator(
    gemma_model,
    gemma_processor,
    train_on_responses_only=True,
    instruction_part=GEMMA_INSTRUCTION_PART,
    response_part=GEMMA_RESPONSE_PART,
    force_match=True,
    completion_only_loss=True,
    resize="max",
)

debug_gemma_batch = gemma_data_collator(
    [
        gemma_train_dataset[0],
        gemma_train_dataset[1],
    ]
)

print("Batch keys:", list(debug_gemma_batch.keys()))
print("Input shape:", debug_gemma_batch["input_ids"].shape)

active_label_tokens = (
    debug_gemma_batch["labels"] != -100
).sum().item()

print("Active label tokens:", active_label_tokens)

assert active_label_tokens > 0, (
    "No assistant tokens were selected for loss."
)

print("Gemma loss masking is ready.")

In [ ]:
# -----------------------------------------ساخت Trainer برای Fine-Tuning Gemma----------------------------------------------

from trl import SFTConfig, SFTTrainer

FastVisionModel.for_training(gemma_model)

USE_BF16 = torch.cuda.is_bf16_supported()

gemma_training_args = SFTConfig(
    output_dir=str(GEMMA_OUTPUT_DIR),

    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type="cosine",

    optim="adamw_8bit",
    weight_decay=0.01,
    max_grad_norm=0.3,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False,
    },

    fp16=not USE_BF16,
    bf16=USE_BF16,

    logging_steps=10,

    save_strategy="steps",
    save_steps=25,
    save_total_limit=2,

    report_to="none",
    remove_unused_columns=False,
    dataset_text_field="",
    dataset_kwargs={
        "skip_prepare_dataset": True,
    },

    max_length=MAX_SEQ_LENGTH,
    seed=SEED,
)

gemma_trainer = SFTTrainer(
    model=gemma_model,
    args=gemma_training_args,
    train_dataset=gemma_train_dataset,
    data_collator=gemma_data_collator,
    processing_class=gemma_processor.tokenizer,
)

print("Gemma trainer is ready.")
print(f"Training rows: {len(gemma_train_dataset)}")
print(
    "Effective batch size: "
    f"{TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}"
)
print(
    "Estimated optimizer steps: "
    f"{len(gemma_train_dataset) // (TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS) * NUM_TRAIN_EPOCHS}"
)
print(f"BF16 enabled: {USE_BF16}")
print(f"Checkpoint directory: {GEMMA_OUTPUT_DIR}")

In [ ]:
# ------------------------------------------آموزش Gemma از آخرین Checkpoint-------------------------------------------------

def get_latest_checkpoint(checkpoint_dir):
    checkpoints = list(Path(checkpoint_dir).glob("checkpoint-*"))

    if not checkpoints:
        return None

    return max(
        checkpoints,
        key=lambda path: int(path.name.split("-")[-1]),
    )


latest_gemma_checkpoint = get_latest_checkpoint(GEMMA_OUTPUT_DIR)

FastVisionModel.for_training(gemma_model)

if latest_gemma_checkpoint is None:
    print("No Gemma checkpoint found. Starting training from step 0.")

    gemma_train_result = gemma_trainer.train()

else:
    print(
        "Resuming Gemma training from: "
        f"{latest_gemma_checkpoint}"
    )

    gemma_train_result = gemma_trainer.train(
        resume_from_checkpoint=str(latest_gemma_checkpoint)
    )

print(gemma_train_result)

In [ ]:
FastVisionModel.for_inference(gemma_model)

gemma_model.save_pretrained(
    str(GEMMA_FINAL_ADAPTER_DIR)
)

gemma_processor.save_pretrained(
    str(GEMMA_FINAL_ADAPTER_DIR)
)

gemma_history_df = pd.DataFrame(
    gemma_trainer.state.log_history
)

gemma_history_path = (
    OUTPUT_ROOT / "gemma_training_history.csv"
)

gemma_history_df.to_csv(
    gemma_history_path,
    index=False,
    encoding="utf-8-sig",
)

print(f"Saved final Gemma LoRA adapter to: {GEMMA_FINAL_ADAPTER_DIR}")
print(f"Saved Gemma training history to: {gemma_history_path}")

In [ ]:
# -----------------------------------------------------------نمودار Loss آموزش Gemma------------------------------------------------------

gemma_loss_history_df = gemma_history_df.dropna(
    subset=["loss"]
).copy()

plt.figure(figsize=(10, 5))

plt.plot(
    gemma_loss_history_df["step"],
    gemma_loss_history_df["loss"],
    marker="o",
)

plt.xlabel("Training step")
plt.ylabel("Training loss")
plt.title("Gemma-3-4B Training Loss")

plt.grid(True)
plt.tight_layout()

gemma_loss_plot_path = (
    OUTPUT_ROOT / "gemma_training_loss.png"
)

plt.savefig(
    gemma_loss_plot_path,
    dpi=200,
    bbox_inches="tight",
)

plt.show()

print(f"Saved training-loss chart to: {gemma_loss_plot_path}")

In [ ]:
# -------------------------------------------------بازسازی تابع پیش‌بینی و ارزیابی Gemma-------------------------------------------

@torch.inference_mode()
def generate_book_title_gemma(
    model,
    processor,
    image,
    max_new_tokens=MAX_NEW_TOKENS,
):
    """
    عنوان کتاب را از روی تصویر جلد با Gemma تولید می‌کند.
    """

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                },
                {
                    "type": "text",
                    "text": TITLE_INSTRUCTION,
                },
            ],
        }
    ]

    prompt = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
    )

    inputs = processor(
        images=[image.convert("RGB")],
        text=[prompt],
        return_tensors="pt",
        padding=True,
    ).to("cuda")

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        use_cache=True,
    )

    new_token_ids = generated_ids[
        :,
        inputs["input_ids"].shape[1]:,
    ]

    raw_prediction = processor.batch_decode(
        new_token_ids,
        skip_special_tokens=True,
    )[0]

    return postprocess_prediction(raw_prediction)


def evaluate_title_model(
    model,
    processor,
    dataset,
    split_name,
    model_label,
    generate_function,
):
    """
    مدل را روی یک split ارزیابی می‌کند و برای هر نمونه
    عنوان واقعی، پیش‌بینی و معیارهای خطا را ذخیره می‌کند.
    """

    rows = []

    for index, example in tqdm(
        enumerate(dataset),
        total=len(dataset),
        desc=f"Evaluating {model_label} on {split_name}",
    ):
        reference_raw = str(example["text"])

        prediction_clean = generate_function(
            model=model,
            processor=processor,
            image=example["image"],
        )

        reference_normalized = normalize_title(reference_raw)
        prediction_normalized = normalize_title(prediction_clean)

        rows.append(
            {
                "index": index,
                "split": split_name,
                "model": model_label,
                "reference_raw": reference_raw,
                "prediction_raw": prediction_clean,
                "prediction_clean": prediction_clean,
                "reference_normalized": reference_normalized,
                "prediction_normalized": prediction_normalized,
                "exact_match": (
                    reference_normalized == prediction_normalized
                ),
                "edit_similarity": normalized_edit_similarity(
                    reference_raw,
                    prediction_clean,
                ),
                "character_error_rate": character_error_rate(
                    reference_raw,
                    prediction_clean,
                ),
            }
        )

    results_df = pd.DataFrame(rows)
    metrics = calculate_metrics(results_df)

    return results_df, metrics


print("Gemma inference and evaluation functions are ready.")

In [ ]:
# -------------------------------------ارزیابی Gemma Fine-Tuned روی Validation---------------------------------------------

FastVisionModel.for_inference(gemma_model)

gemma_finetuned_val_df, gemma_finetuned_val_metrics = (
    evaluate_title_model(
        model=gemma_model,
        processor=gemma_processor,
        dataset=val_raw,
        split_name="validation",
        model_label="Gemma-3-4B-Fine-Tuned",
        generate_function=generate_book_title_gemma,
    )
)

display(pd.DataFrame([gemma_finetuned_val_metrics]))

gemma_finetuned_val_path = (
    OUTPUT_ROOT / "gemma_finetuned_validation_predictions.csv"
)

gemma_finetuned_val_df.to_csv(
    gemma_finetuned_val_path,
    index=False,
    encoding="utf-8-sig",
)

print(
    "Saved Gemma fine-tuned validation predictions to: "
    f"{gemma_finetuned_val_path}"
)

In [ ]:
# -------------------------------------------------مقایسه‌ی Gemma قبل و بعد از Fine-Tuning------------------------------------------------

gemma_baseline_val_path = (
    OUTPUT_ROOT / "gemma_baseline_validation_predictions.csv"
)

gemma_baseline_val_df = pd.read_csv(
    gemma_baseline_val_path
)

gemma_baseline_val_metrics = calculate_metrics(
    gemma_baseline_val_df
)

gemma_validation_comparison_df = pd.DataFrame(
    [
        {
            "Model": "Gemma-3-4B Base",
            **gemma_baseline_val_metrics,
        },
        {
            "Model": "Gemma-3-4B Fine-Tuned",
            **gemma_finetuned_val_metrics,
        },
    ]
)

base_row = gemma_validation_comparison_df.iloc[0]

gemma_validation_comparison_df["Exact Match Change (pp)"] = (
    gemma_validation_comparison_df["exact_match_percent"]
    - base_row["exact_match_percent"]
)

gemma_validation_comparison_df["Edit Similarity Change (pp)"] = (
    gemma_validation_comparison_df[
        "mean_edit_similarity_percent"
    ]
    - base_row["mean_edit_similarity_percent"]
)

gemma_validation_comparison_df["CER Change (pp)"] = (
    gemma_validation_comparison_df[
        "mean_character_error_rate_percent"
    ]
    - base_row["mean_character_error_rate_percent"]
)

display(gemma_validation_comparison_df)

gemma_validation_comparison_path = (
    OUTPUT_ROOT / "gemma_validation_comparison.csv"
)

gemma_validation_comparison_df.to_csv(
    gemma_validation_comparison_path,
    index=False,
    encoding="utf-8-sig",
)

print(
    "Saved Gemma validation comparison to: "
    f"{gemma_validation_comparison_path}"
)

In [ ]:
# ---------------------------------------ثبت خطاها و بهترین بهبودهای Gemma در Validation-----------------------------------------------

gemma_finetuned_errors_df = (
    gemma_finetuned_val_df[
        ~gemma_finetuned_val_df["exact_match"]
    ]
    .sort_values(
        by=["edit_similarity", "character_error_rate"],
        ascending=[True, False],
    )
    .copy()
)

gemma_finetuned_errors_path = (
    OUTPUT_ROOT / "gemma_finetuned_validation_errors.csv"
)

gemma_finetuned_errors_df.to_csv(
    gemma_finetuned_errors_path,
    index=False,
    encoding="utf-8-sig",
)

gemma_before_after_df = gemma_baseline_val_df.merge(
    gemma_finetuned_val_df,
    on="index",
    suffixes=("_base", "_finetuned"),
)

gemma_before_after_df["similarity_gain"] = (
    gemma_before_after_df["edit_similarity_finetuned"]
    - gemma_before_after_df["edit_similarity_base"]
)

gemma_before_after_df["cer_reduction"] = (
    gemma_before_after_df["character_error_rate_base"]
    - gemma_before_after_df["character_error_rate_finetuned"]
)

gemma_before_after_df = gemma_before_after_df.sort_values(
    by=["similarity_gain", "cer_reduction"],
    ascending=[False, False],
).copy()

gemma_before_after_path = (
    OUTPUT_ROOT / "gemma_validation_before_after.csv"
)

gemma_before_after_df.to_csv(
    gemma_before_after_path,
    index=False,
    encoding="utf-8-sig",
)

display(
    gemma_before_after_df[
        [
            "index",
            "reference_raw_base",
            "prediction_clean_base",
            "prediction_clean_finetuned",
            "edit_similarity_base",
            "edit_similarity_finetuned",
            "similarity_gain",
            "character_error_rate_base",
            "character_error_rate_finetuned",
        ]
    ]
    .head(10)
    .reset_index(drop=True)
)

display(
    gemma_finetuned_errors_df[
        [
            "index",
            "reference_raw",
            "prediction_clean",
            "edit_similarity",
            "character_error_rate",
        ]
    ]
    .head(6)
    .reset_index(drop=True)
)

print(
    "Saved Gemma fine-tuned errors to: "
    f"{gemma_finetuned_errors_path}"
)

print(
    "Saved Gemma before/after comparison to: "
    f"{gemma_before_after_path}"
)

## 6. Held-Out Test Evaluation

After both training configurations were finalized, the base and fine-tuned versions of each model were evaluated on the same untouched test split of 250 samples.

No test-set result was used to adjust hyperparameters, prompts, or training settings.


In [ ]:
# ---------------------------------------ارزیابی  Gemma Fine-Tuned روی Test مستقل---------------------------------------

FastVisionModel.for_inference(gemma_model)

gemma_finetuned_test_df, gemma_finetuned_test_metrics = (
    evaluate_title_model(
        model=gemma_model,
        processor=gemma_processor,
        dataset=test_raw,
        split_name="test",
        model_label="Gemma-3-4B-Fine-Tuned",
        generate_function=generate_book_title_gemma,
    )
)

display(pd.DataFrame([gemma_finetuned_test_metrics]))

gemma_finetuned_test_path = (
    OUTPUT_ROOT / "gemma_finetuned_test_predictions.csv"
)

gemma_finetuned_test_df.to_csv(
    gemma_finetuned_test_path,
    index=False,
    encoding="utf-8-sig",
)

print(
    "Saved Gemma fine-tuned test predictions to: "
    f"{gemma_finetuned_test_path}"
)

In [ ]:
for variable_name in [
    "gemma_model",
    "gemma_processor",
    "gemma_trainer",
    "gemma_data_collator",
    "debug_gemma_batch",
]:
    if variable_name in globals():
        del globals()[variable_name]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

show_gpu_memory()

In [ ]:
# ---------------------------------------------------ارگذاری Gemma پایه برای ارزیابی نهایی Test-------------------------------------------------

from unsloth import FastVisionModel, get_chat_template

gemma_base_model, gemma_base_processor = (
    FastVisionModel.from_pretrained(
        model_name=GEMMA_MODEL_ID,
        load_in_4bit=True,
        use_gradient_checkpointing="unsloth",
        max_seq_length=MAX_SEQ_LENGTH,
    )
)

gemma_base_processor = get_chat_template(
    gemma_base_processor,
    "gemma-3",
)

FastVisionModel.for_inference(gemma_base_model)

show_gpu_memory()

print("Base Gemma is ready for final test evaluation.")

In [ ]:
# ------------------------------------------ارزیابی Gemma پایه روی Test مستقل------------------------------------

gemma_base_test_df, gemma_base_test_metrics = (
    evaluate_title_model(
        model=gemma_base_model,
        processor=gemma_base_processor,
        dataset=test_raw,
        split_name="test",
        model_label="Gemma-3-4B-Base",
        generate_function=generate_book_title_gemma,
    )
)

display(pd.DataFrame([gemma_base_test_metrics]))

gemma_base_test_path = (
    OUTPUT_ROOT / "gemma_base_test_predictions.csv"
)

gemma_base_test_df.to_csv(
    gemma_base_test_path,
    index=False,
    encoding="utf-8-sig",
)

print(
    "Saved Gemma base test predictions to: "
    f"{gemma_base_test_path}"
)

In [ ]:
# ------------------------------------------- جدول  Gemma روی Test------------------------------------------------

gemma_base_test_df = pd.read_csv(
    OUTPUT_ROOT / "gemma_base_test_predictions.csv"
)

gemma_finetuned_test_df = pd.read_csv(
    OUTPUT_ROOT / "gemma_finetuned_test_predictions.csv"
)

gemma_base_test_metrics = calculate_metrics(gemma_base_test_df)
gemma_finetuned_test_metrics = calculate_metrics(gemma_finetuned_test_df)

gemma_test_comparison_df = pd.DataFrame(
    [
        {"Model": "Gemma-3-4B Base", **gemma_base_test_metrics},
        {"Model": "Gemma-3-4B Fine-Tuned", **gemma_finetuned_test_metrics},
    ]
)

base_row = gemma_test_comparison_df.iloc[0]

gemma_test_comparison_df["Exact Match Change (pp)"] = (
    gemma_test_comparison_df["exact_match_percent"]
    - base_row["exact_match_percent"]
)

gemma_test_comparison_df["Edit Similarity Change (pp)"] = (
    gemma_test_comparison_df["mean_edit_similarity_percent"]
    - base_row["mean_edit_similarity_percent"]
)

gemma_test_comparison_df["CER Change (pp)"] = (
    gemma_test_comparison_df["mean_character_error_rate_percent"]
    - base_row["mean_character_error_rate_percent"]
)

display(gemma_test_comparison_df)

gemma_test_comparison_path = OUTPUT_ROOT / "gemma_test_comparison.csv"

gemma_test_comparison_df.to_csv(
    gemma_test_comparison_path,
    index=False,
    encoding="utf-8-sig",
)

print(f"Saved final Gemma test comparison to: {gemma_test_comparison_path}")

In [ ]:
# -----------------------------------------------------------بارگذاری Qwen پایه برای ارزیابی Test-------------------------------------

for variable_name in [
    "gemma_base_model",
    "gemma_base_processor",
    "gemma_model",
    "gemma_processor",
]:
    if variable_name in globals():
        del globals()[variable_name]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

QWEN_MODEL_ID = "unsloth/Qwen2.5-VL-3B-Instruct-bnb-4bit"
QWEN_FINAL_ADAPTER_DIR = (
    OUTPUT_ROOT / "qwen25vl3b_persian_book_title_lora"
)

assert QWEN_FINAL_ADAPTER_DIR.exists(), (
    f"Qwen adapter was not found: {QWEN_FINAL_ADAPTER_DIR}"
)

qwen_base_model, qwen_processor = FastVisionModel.from_pretrained(
    model_name=QWEN_MODEL_ID,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
    max_seq_length=MAX_SEQ_LENGTH,
)

FastVisionModel.for_inference(qwen_base_model)


@torch.inference_mode()
def generate_book_title_qwen(
    model,
    processor,
    image,
    max_new_tokens=MAX_NEW_TOKENS,
):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_MESSAGE,
        },
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": TITLE_INSTRUCTION},
            ],
        },
    ]

    prompt = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = processor(
        images=[image.convert("RGB")],
        text=[prompt],
        return_tensors="pt",
        padding=True,
    ).to("cuda")

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        use_cache=True,
    )

    new_token_ids = generated_ids[
        :,
        inputs["input_ids"].shape[1]:,
    ]

    raw_prediction = processor.batch_decode(
        new_token_ids,
        skip_special_tokens=True,
    )[0]

    return postprocess_prediction(raw_prediction)


show_gpu_memory()
print("Qwen base model is ready for final test evaluation.")

In [ ]:
# -----------------------------------------سلول 68 — ارزیابی Qwen پایه روی Test مستقل---------------------------------------

qwen_base_test_df, qwen_base_test_metrics = (
    evaluate_title_model(
        model=qwen_base_model,
        processor=qwen_processor,
        dataset=test_raw,
        split_name="test",
        model_label="Qwen2.5-VL-3B-Base",
        generate_function=generate_book_title_qwen,
    )
)

display(pd.DataFrame([qwen_base_test_metrics]))

qwen_base_test_path = OUTPUT_ROOT / "qwen_base_test_predictions.csv"

qwen_base_test_df.to_csv(
    qwen_base_test_path,
    index=False,
    encoding="utf-8-sig",
)

print(
    "Saved Qwen base test predictions to: "
    f"{qwen_base_test_path}"
)

In [ ]:
# ----------------------------------------------------بارگذاری Qwen Fine-Tuned از آداپتر---------------------------------------------

for variable_name in [
    "qwen_base_model",
    "qwen_processor",
]:
    if variable_name in globals():
        del globals()[variable_name]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

qwen_finetuned_model, qwen_finetuned_processor = (
    FastVisionModel.from_pretrained(
        model_name=str(QWEN_FINAL_ADAPTER_DIR),
        load_in_4bit=True,
        use_gradient_checkpointing="unsloth",
        max_seq_length=MAX_SEQ_LENGTH,
    )
)

FastVisionModel.for_inference(qwen_finetuned_model)

show_gpu_memory()
print("Qwen fine-tuned LoRA adapter is ready for final test evaluation.")

In [ ]:
# -------------------------------------ارزیابی Qwen Fine-Tuned روی Test مستقل---------------------------------------------

qwen_finetuned_test_df, qwen_finetuned_test_metrics = (
    evaluate_title_model(
        model=qwen_finetuned_model,
        processor=qwen_finetuned_processor,
        dataset=test_raw,
        split_name="test",
        model_label="Qwen2.5-VL-3B-Fine-Tuned",
        generate_function=generate_book_title_qwen,
    )
)

display(pd.DataFrame([qwen_finetuned_test_metrics]))

qwen_finetuned_test_path = (
    OUTPUT_ROOT / "qwen_finetuned_test_predictions.csv"
)

qwen_finetuned_test_df.to_csv(
    qwen_finetuned_test_path,
    index=False,
    encoding="utf-8-sig",
)

print(
    "Saved Qwen fine-tuned test predictions to: "
    f"{qwen_finetuned_test_path}"
)

## 7. Final Comparison and Model Selection

The following cell consolidates the held-out test metrics across all four evaluated configurations and produces the final comparison chart.


In [ ]:
test_result_paths = {
    "Qwen2.5-VL-3B Base": OUTPUT_ROOT / "qwen_base_test_predictions.csv",
    "Qwen2.5-VL-3B Fine-Tuned": (
        OUTPUT_ROOT / "qwen_finetuned_test_predictions.csv"
    ),
    "Gemma-3-4B Base": OUTPUT_ROOT / "gemma_base_test_predictions.csv",
    "Gemma-3-4B Fine-Tuned": (
        OUTPUT_ROOT / "gemma_finetuned_test_predictions.csv"
    ),
}

final_test_rows = []

for model_name, file_path in test_result_paths.items():
    predictions_df = pd.read_csv(file_path)
    metrics = calculate_metrics(predictions_df)

    final_test_rows.append(
        {
            "Model": model_name,
            **metrics,
        }
    )

final_test_results_df = pd.DataFrame(final_test_rows)

final_test_results_df = final_test_results_df[
    [
        "Model",
        "exact_match_percent",
        "mean_edit_similarity_percent",
        "mean_character_error_rate_percent",
    ]
].rename(
    columns={
        "exact_match_percent": "Exact Match (%)",
        "mean_edit_similarity_percent": "Edit Similarity (%)",
        "mean_character_error_rate_percent": "CER (%)",
    }
)

display(final_test_results_df)

final_test_results_path = OUTPUT_ROOT / "final_test_results_four_models.csv"

final_test_results_df.to_csv(
    final_test_results_path,
    index=False,
    encoding="utf-8-sig",
)

qwen_base_row = final_test_results_df[
    final_test_results_df["Model"] == "Qwen2.5-VL-3B Base"
].iloc[0]

qwen_finetuned_row = final_test_results_df[
    final_test_results_df["Model"] == "Qwen2.5-VL-3B Fine-Tuned"
].iloc[0]

gemma_base_row = final_test_results_df[
    final_test_results_df["Model"] == "Gemma-3-4B Base"
].iloc[0]

gemma_finetuned_row = final_test_results_df[
    final_test_results_df["Model"] == "Gemma-3-4B Fine-Tuned"
].iloc[0]

fine_tuning_changes_df = pd.DataFrame(
    [
        {
            "Model Family": "Qwen2.5-VL-3B",
            "Exact Match Change (pp)": (
                qwen_finetuned_row["Exact Match (%)"]
                - qwen_base_row["Exact Match (%)"]
            ),
            "Edit Similarity Change (pp)": (
                qwen_finetuned_row["Edit Similarity (%)"]
                - qwen_base_row["Edit Similarity (%)"]
            ),
            "CER Change (pp)": (
                qwen_finetuned_row["CER (%)"]
                - qwen_base_row["CER (%)"]
            ),
        },
        {
            "Model Family": "Gemma-3-4B",
            "Exact Match Change (pp)": (
                gemma_finetuned_row["Exact Match (%)"]
                - gemma_base_row["Exact Match (%)"]
            ),
            "Edit Similarity Change (pp)": (
                gemma_finetuned_row["Edit Similarity (%)"]
                - gemma_base_row["Edit Similarity (%)"]
            ),
            "CER Change (pp)": (
                gemma_finetuned_row["CER (%)"]
                - gemma_base_row["CER (%)"]
            ),
        },
    ]
)

display(fine_tuning_changes_df)

fine_tuning_changes_path = (
    OUTPUT_ROOT / "final_test_finetuning_changes.csv"
)

fine_tuning_changes_df.to_csv(
    fine_tuning_changes_path,
    index=False,
    encoding="utf-8-sig",
)

plot_df = final_test_results_df.copy()

metrics_to_plot = [
    "Exact Match (%)",
    "Edit Similarity (%)",
    "CER (%)",
]

model_colors = {
    "Qwen2.5-VL-3B Base": "#89BFFF",
    "Qwen2.5-VL-3B Fine-Tuned": "#155AA8",
    "Gemma-3-4B Base": "#FF9EB5",
    "Gemma-3-4B Fine-Tuned": "#B51F52",
}

x = np.arange(len(metrics_to_plot))
bar_width = 0.18

plt.figure(figsize=(11, 5.5))

for row_index, (_, row) in enumerate(plot_df.iterrows()):
    offsets = x + (row_index - 1.5) * bar_width
    values = [row[metric_name] for metric_name in metrics_to_plot]

    bars = plt.bar(
        offsets,
        values,
        width=bar_width,
        label=row["Model"],
        color=model_colors[row["Model"]],
        edgecolor="#444444",
        linewidth=0.35,
    )

    for bar, value in zip(bars, values):
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.7,
            f"{value:.1f}",
            ha="center",
            va="bottom",
            fontsize=8,
        )

plt.xticks(
    x,
    [
        "Exact Match ↑",
        "Edit Similarity ↑",
        "CER ↓",
    ],
)

plt.ylabel("Percentage")
plt.title("Final Held-Out Test Results: Base vs Fine-Tuned VLMs")
plt.ylim(0, 75)
plt.legend()
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()

final_test_plot_path = (
    OUTPUT_ROOT / "final_test_metrics_compact_soft.png"
)

plt.savefig(
    final_test_plot_path,
    dpi=180,
    bbox_inches="tight",
)

plt.show()

print(f"Saved four-model test results to: {final_test_results_path}")
print(f"Saved fine-tuning changes to: {fine_tuning_changes_path}")
print(f"Saved final test chart to: {final_test_plot_path}")

## 8. Final Outcome

On the held-out test set, **Gemma-3-4B Fine-Tuned** achieved the strongest overall result:

| Model | Exact Match (%) | Edit Similarity (%) | CER (%) |
|---|---:|---:|---:|
| Qwen2.5-VL-3B Base | 6.4 | 43.96 | 59.51 |
| Qwen2.5-VL-3B Fine-Tuned | 12.0 | 53.54 | 63.17 |
| Gemma-3-4B Base | 8.0 | 41.15 | 60.00 |
| **Gemma-3-4B Fine-Tuned** | **14.8** | **55.16** | **53.09** |

Gemma-3-4B Fine-Tuned is selected as the final model because it achieved the highest Exact Match and Edit Similarity, together with the lowest CER.

### Known Limitations

Remaining errors often involve artistic typography, long titles, mixed Persian/English cover text, and minor word or volume-number mistakes.
